<a href="https://colab.research.google.com/github/DiyaRana7/Flyrank_ML/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1 — Growing content has a different structural profile

The paper reports that growing pages tend to be longer, younger, and slightly better positioned than declining pages. In the reported comparison, growing content averaged about 3.2K words and 184 days of age, while declining content averaged about 2.3K words and 230 days of age.

**Methodology question:** How exactly was the growing/declining label defined, and was the information used to define that label kept separate from the features being compared? I would also want to confirm that the comparison is descriptive rather than evidence that age or word count causes growth.

The question is constructive because the finding can still be useful as an observed relationship even if the underlying factors are confounded.

### Finding 2 — High-AI pages behave differently from no-AI pages

The paper reports that pages with high AI-referral activity had much higher average impressions but weaker average Google positions than pages with no AI referrals. The paper interprets this as evidence that AI-referral visibility is not simply a mirror of Google ranking.

**Methodology question:** How was the AI-referral bucket defined, and does the analysis control for differences in content age, visibility, or other factors that could affect both AI referrals and search performance? I would also check whether the small AI-traffic base is large enough for stable conclusions across buckets.

The finding is useful as a directional observation, but the methodology should make clear that the relationship is not causal.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

### Before/after validation comparison

The original Week-5 model used a grouped-by-client split with a 33% test set and random seed 42.

For this audit, I re-ran the model using the same grouped-by-client design. The test set contains 11,310 rows from 11 clients, while the training set contains 18,690 rows from 21 clients.

The training and test declining rates are similar (54.6% and 53.6%), and there is no client overlap between the two sets.

This supports the split as a reasonable grouped validation design for this dataset. It does not prove that the model will generalize to all future data.

In [14]:
from sklearn.model_selection import GroupShuffleSplit

# Use the same target as Week 5
y = (df["trend_direction"] == "down").astype(int)

# Use the same feature set as Week 5
feature_cols = [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

X = df[feature_cols].copy()
X = X.fillna(X.median(numeric_only=True))

groups = df["client_id"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.33,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

print("Training rows:", len(train_idx))
print("Test rows:", len(test_idx))

print("\nTraining clients:",
      df.iloc[train_idx]["client_id"].nunique())

print("Test clients:",
      df.iloc[test_idx]["client_id"].nunique())

print("\nTraining declining rate:",
      round(y.iloc[train_idx].mean(), 3))

print("Test declining rate:",
      round(y.iloc[test_idx].mean(), 3))

# Check that no client appears in both sets
train_clients = set(df.iloc[train_idx]["client_id"])
test_clients = set(df.iloc[test_idx]["client_id"])

overlap = train_clients.intersection(test_clients)

print("\nClient overlap:", overlap)

if len(overlap) == 0:
    print("PASS - no client appears in both train and test.")
else:
    print("FAIL - client leakage detected.")

Training rows: 18690
Test rows: 11310

Training clients: 21
Test clients: 11

Training declining rate: 0.546
Test declining rate: 0.536

Client overlap: set()
PASS - no client appears in both train and test.


### Before/after validation comparison

The original Week-5 evaluation used a grouped-by-client split. For this audit, the same model and feature set are evaluated under the honest grouped split to check whether performance remains stable when clients are separated between training and testing.

The comparison is intended to show whether the model's measured performance depends strongly on the validation design. The grouped split is more conservative because the same client cannot appear in both training and test sets.


In [3]:
import pandas as pd
import numpy as np

from datasets import load_dataset
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

daily = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    split="train[:30000]",
    token=HF_TOKEN
)

df = daily.to_pandas()

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  624kB            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 3.22MB            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 4.41MB            

...

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  146MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/78835655 [00:00<?, ? examples/s]

Dataset shape: (30000, 30)
Columns: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']


In [4]:
print("Columns containing 'label':")
print([c for c in df.columns if "label" in c.lower()])

print("\nColumns containing 'trend':")
print([c for c in df.columns if "trend" in c.lower()])

Columns containing 'label':
[]

Columns containing 'trend':
[]


In [8]:
!git clone https://github.com/DiyaRana7/Flyrank_ML.git /content/Flyrank_ML

Cloning into '/content/Flyrank_ML'...
remote: Enumerating objects: 129, done.
remote: Counting objects: 100% (129/129), done.
remote: Compressing objects: 100% (100/100), done.
remote: Total 129 (delta 41), reused 79 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (129/129), 1.87 MiB | 9.12 MiB/s, done.
Resolving deltas: 100% (41/41), done.


In [10]:
!pwd
!find /content -name "w05_model.ipynb" 2>/dev/null

/content
/content/Flyrank_ML/work/notebooks/w05_model.ipynb


In [9]:
import json

path = "/content/Flyrank_ML/work/notebooks/w05_model.ipynb"

with open(path, "r", encoding="utf-8") as f:
    nb = json.load(f)

for i, cell in enumerate(nb["cells"]):
    if cell["cell_type"] == "code":
        code = "".join(cell["source"])
        print(f"\n--- CODE CELL {i+1} ---\n")
        print(code)


--- CODE CELL 5 ---

!git clone https://github.com/DiyaRana7/Flyrank_ML.git
%cd Flyrank_ML

--- CODE CELL 6 ---

import pandas as pd
import numpy as np
import os

# Make sure we are inside the repository
REPO_PATH = "/content/Flyrank_ML"

if os.path.exists(REPO_PATH):
    os.chdir(REPO_PATH)
else:
    raise FileNotFoundError(
        "Flyrank_ML repository not found. Clone the repository first."
    )

# Load the starter dataset used for the baseline
DATA_PATH = "data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

--- CODE CELL 7 ---

candidate_features = [
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "engagement_rate",
    "scroll_rate",
    "word_count",
    "char_count",
    "search_volume",
    "competition",
    "cpc"
]

available_features = [
    col for col in candidate_features
    if col in df.columns
]

print("Available candidate fea

### Leakage audit

I checked the final feature set for fields that could directly contain the outcome or information derived from the outcome.

In particular, `trend_direction` and `trend_pct` are excluded from the feature matrix because they describe observed trend outcomes and could leak information about the target.

`content_id` and `client_id` are also excluded as predictive features. They are used only for identification and grouped validation.

The model therefore uses performance, content, freshness, and engagement signals available in the dataset rather than the target itself.

In [15]:
# Final feature leakage audit

target_column = "trend_direction"

leakage_candidates = [
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "content_id",
    "client_id"
]

print("Target:", target_column)

print("\nFinal model features:")
print(feature_cols)

print("\nPotential leakage / identifier fields:")
for col in leakage_candidates:
    if col in feature_cols:
        print(f"FAIL - {col} is included in model features")
    else:
        print(f"PASS - {col} is not included in model features")

# Check for suspicious feature names
suspicious_terms = [
    "label",
    "target",
    "trend"
]

suspicious_features = [
    col for col in feature_cols
    if any(term in col.lower() for term in suspicious_terms)
]

print("\nFeatures with suspicious label/target/trend names:")
print(suspicious_features)

if len(suspicious_features) == 0:
    print("PASS - no suspicious label/target/trend feature names found.")
else:
    print("REVIEW - inspect the listed features manually.")

Target: trend_direction

Final model features:
['impressions_90d', 'clicks_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']

Potential leakage / identifier fields:
PASS - trend_direction is not included in model features
PASS - trend_pct is not included in model features
PASS - is_declining_label is not included in model features
PASS - content_id is not included in model features
PASS - client_id is not included in model features

Features with suspicious label/target/trend names:
[]
PASS - no suspicious label/target/trend feature names found.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## 4. Claim rewrite

### Original claim

The Random Forest model can predict which content pages are declining and identify pages that should be refreshed.

### Safer claim

On the evaluated dataset, the Random Forest model showed measured predictive performance for the selected declining-trend target under a client-grouped validation split. The results are directional and support decision-making about which pages may deserve further review. They do not establish that the model can reliably predict future performance for all pages, nor do they show that refreshing a page will cause its performance to improve.

In [16]:
print("Claim audit:")
print()

print("Original claim:")
print(
    "The Random Forest model can identify pages that are likely "
    "to experience declining search performance and can be used "
    "to prioritize content refresh decisions."
)

print("\nSafer claim:")
print(
    "On the evaluated grouped client split, the Random Forest model "
    "showed measured performance for identifying pages associated "
    "with the observed trend_direction outcome. The results are "
    "directional and support using the model as decision-support "
    "for prioritizing pages for human review. They do not establish "
    "that the model predicts future Google rankings or that refreshing "
    "a page will cause its performance to improve."
)

Claim audit:

Original claim:
The Random Forest model can identify pages that are likely to experience declining search performance and can be used to prioritize content refresh decisions.

Safer claim:
On the evaluated grouped client split, the Random Forest model showed measured performance for identifying pages associated with the observed trend_direction outcome. The results are directional and support using the model as decision-support for prioritizing pages for human review. They do not establish that the model predicts future Google rankings or that refreshing a page will cause its performance to improve.


In [17]:
print("ML-09 final audit")
print("----------------")

print("Grouped validation: PASS")
print("Client overlap:", set(df.iloc[train_idx]["client_id"]) &
      set(df.iloc[test_idx]["client_id"]))

print("Leakage audit: PASS")
print("Target used as feature:", "trend_direction" in feature_cols)
print("Trend percentage used as feature:", "trend_pct" in feature_cols)
print("Content ID used as feature:", "content_id" in feature_cols)
print("Client ID used as feature:", "client_id" in feature_cols)

ML-09 final audit
----------------
Grouped validation: PASS
Client overlap: set()
Leakage audit: PASS
Target used as feature: False
Trend percentage used as feature: False
Content ID used as feature: False
Client ID used as feature: False


## Self-check

- [x] Every section is filled with markdown and supporting code.
- [x] The notebook runs top to bottom without errors.
- [x] The model uses grouped-by-client validation with no client overlap.
- [x] Training and test declining rates are reasonably similar.
- [x] `trend_direction` and `trend_pct` are excluded from model features.
- [x] Client and content IDs are not used as predictive features.
- [x] Real error examples were inspected.
- [x] Claims use careful language such as observed, measured, directional, and decision-support.
- [x] No client names, URLs, private queries, or credentials are included.
- [x] The notebook is ready to commit under `work/notebooks/w06_validation_audit.ipynb`.